Step 1 <br>

We will be focusing on fetching youtube video's url , we have to find only Ekantik videos from Bhajan Marg youtube channel. This is done via yt-dl from terminal. read my documentation for step by step procedure or simply run 
<br>(pyTensor) tejas@Tejass-MacBook-Air Ekantik Project % yt-dlp \
  --flat-playlist \
  --dump-json \
  "https://www.youtube.com/playlist?list=PLNlLlnQWoRlndo0jrtUb6XmJJdnOaMUw5" \
  > ekantik_playlist_raw.jsonl
<br>
in terminal

Now is to filter out the fetched playlist, we have removed unnecessary data like {thumbnails,view_count,uploader / channel metadata,extractor info,epoch,availability,playlist metadata duplicates,description,live_status,version info} <br>

We want "video_id","title","url","duration_sec","declared_ekantik_number"

In [13]:
import json
import re
import os

INPUT = "ekantik_playlist_raw.jsonl"
OUTPUT = "ekantik_videos_v1.json"

if os.path.exists(OUTPUT): # to keep a check if the file already exists
    print(f"{OUTPUT} already exists.")
else:
    # Match title starting with "#<number>"
    ekantik_number_regex = re.compile(r"^\s*#\s*(\d+)")

    videos = []
    removed = 0

    with open(INPUT, "r", encoding="utf-8") as f:
        for line in f:
            raw = json.loads(line)

            title = raw.get("title")

            # Skip private / deleted
            if title in ("[Private video]", "[Deleted video]"):
                removed += 1
                continue

            # Skip inaccessible videos
            if raw.get("duration") is None:
                removed += 1
                continue

            match = ekantik_number_regex.search(title)
            declared_number = int(match.group(1)) if match else None

            videos.append({
                "video_id": raw["id"],
                "title": title,
                "url": raw.get("webpage_url"),
                "duration_sec": raw.get("duration"),
                "declared_ekantik_number": declared_number
            })

    with open(OUTPUT, "w", encoding="utf-8") as f:
        json.dump(videos, f, ensure_ascii=False, indent=2)

    print(f"Saved {len(videos)} Ekantik videos → {OUTPUT}")
    print(f"Removed {removed} private/inaccessible videos")
    print(
        "Videos with declared Ekantik number:",
        sum(1 for v in videos if v["declared_ekantik_number"] is not None)
    )


ekantik_videos_v1.json already exists.


**Phase 2 : Fetching the Transcript**

1. Fetch both english and hindi transcripts 


So each of Ekantik has a  "video_id": "wGW7GKGbg9I" and "declared_ekantik_number": 21 we use this information to name each .json if we  get "declared_ekantik_number": null we will use video_id to name the file

Defining Naming convention and code to store transcript cleanly

In [14]:
def get_page_name(video_id, declared_ekantik_number):
    if declared_ekantik_number is not None:
        return f"ekantik_{declared_ekantik_number}.json"
    return f"video_{video_id}.json"


def save_json(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


To fetch transcript in hindi

In [29]:
from youtube_transcript_api import YouTubeTranscriptApi,TranscriptsDisabled,NoTranscriptFound

def fetch_hindi_transcript(video_id):
    transcript_list = YouTubeTranscriptApi().fetch(video_id, languages=["hi"])

    return transcript_list


Structure Document from fetched transcript / snippet

In [36]:
def build_page(video_id, declared_ekantik_number, snippets):
    return {
        "video_id": video_id,
        "declared_ekantik_number": declared_ekantik_number,
        "language": "hi",
        "snippets": [
            {
                "index": i + 1,
                "text": s.text,
                "start": s.start,
                "duration": s.duration,
            }
            for i, s in enumerate(snippets)
        ],
    }


Processing each page using : build_page,fetch_hindi_transcript,save_json,get_page_name

In [38]:
from pathlib import Path
def process_video(entry, failures):
    video_id = entry["video_id"]
    ekantik_no = entry.get("declared_ekantik_number")

    page_name = get_page_name(video_id, ekantik_no)
    output_path = Path("transcripts/hindi") / page_name


    if output_path.exists():
        print(f"already processed {page_name}")
        return  # already processed, skip

    try:
        snippets = fetch_hindi_transcript(video_id)
        page = build_page(video_id, ekantik_no, snippets)

        save_json(output_path, page)

    except (TranscriptsDisabled, NoTranscriptFound):
        failures.append({
            "video_id": video_id,
            "declared_ekantik_number": ekantik_no,
            "reason": "Hindi transcript not available"
        })


Process all videos from our ekantik_video.json

In [18]:
def generate_transcripts(video_meta_data):
    failures = []

    for entry in video_meta_data:
        process_video(entry, failures)

    save_json(
        Path("transcripts/transcript_failures.json"),
        failures
    )


Phase 2 : Getting all the transcripts

In [42]:
with open("ekantik_videos_v1.json", "r", encoding="utf-8") as f:
    video_meta_data = json.load(f)

generate_transcripts(video_meta_data)

already processed ekantik_1138.json
already processed ekantik_1137.json
already processed ekantik_1136.json
already processed ekantik_1135.json
already processed ekantik_1134.json
already processed ekantik_1133.json
already processed ekantik_1132.json
already processed ekantik_1131.json
already processed ekantik_1130.json
already processed ekantik_1129.json
already processed ekantik_1128.json
already processed ekantik_1127.json
already processed ekantik_1126.json
already processed ekantik_1125.json
already processed ekantik_1124.json
already processed ekantik_1123.json
already processed ekantik_1122.json
already processed ekantik_1121.json
already processed ekantik_1120.json
already processed ekantik_1119.json
already processed ekantik_1118.json
already processed ekantik_1117.json
already processed ekantik_1116.json
already processed ekantik_1115.json
already processed ekantik_1114.json
already processed ekantik_1113.json
already processed ekantik_1112.json
already processed ekantik_11

ConnectTimeout: HTTPSConnectionPool(host='www.youtube.com', port=443): Max retries exceeded with url: /watch?v=QkrHHz02BNk (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x10bf96200>, 'Connection to www.youtube.com timed out. (connect timeout=None)'))

In [ ]:
transcript_list = YouTubeTranscriptApi().fetch("51Fhwt3bDzg", languages=["hi"])

In [35]:
def build_page(video_id, declared_ekantik_number, snippets):
    return {
        "video_id": video_id,
        "declared_ekantik_number": declared_ekantik_number,
        "language": "hi",
        "snippets": [
            {
                "index": i + 1,
                "text": s.text,
                "start": s.start,
                "duration": s.duration,
            }
            for i, s in enumerate(snippets)
        ],
    }

result = build_page("51Fhwt3bDzg",20,transcript_list)
print(result)

{'video_id': '51Fhwt3bDzg', 'declared_ekantik_number': 20, 'language': 'hi', 'snippets': [{'index': 1, 'text': '[संगीत]', 'start': 2.13, 'duration': 10.88}, {'index': 2, 'text': '[हंसी]', 'start': 10.62, 'duration': 4.66}, {'index': 3, 'text': '[संगीत]', 'start': 13.01, 'duration': 6.429}, {'index': 4, 'text': 'गणेश शर्मा जी मध्य प्रदेश से परम पूज्य', 'start': 15.28, 'duration': 6.32}, {'index': 5, 'text': 'महाराज जी के चरणों में दंडवत प्रणाम।', 'start': 19.439, 'duration': 5.281}, {'index': 6, 'text': 'गुरु जी परिवार के पास कुछ दिन रहने', 'start': 21.6, 'duration': 5.999}, {'index': 7, 'text': 'वृंदावन आया था किंतु सत्संग श्रवण एवं', 'start': 24.72, 'duration': 5.44}, {'index': 8, 'text': 'परिक्रमा करते दो माह हो गए। आपके दर्शन', 'start': 27.599, 'duration': 5.12}, {'index': 9, 'text': 'से हृदय में बहुत आनंद होता है। पर मन कई', 'start': 30.16, 'duration': 5.6}, {'index': 10, 'text': 'बार वापस जाकर नौकरी आदि को आंदोलित करता', 'start': 32.719, 'duration': 6.801}, {'index': 11, 'text': '